In [1]:
import pandas as pd
import numpy as np
import ops

print(pd.__version__)

2.3.3


In [2]:
N = 10_000_000
np.random.seed(1234)  # set seed for reproducibility
arr = np.random.rand(N) * 100

df = pd.DataFrame({"x": arr})
df

,x
0,19.151945
1,62.210877
2,43.772774
3,78.535858
4,77.997581
...,...
9999995,11.115537
9999996,49.682310
9999997,19.468962
9999998,82.011629


In [3]:
window = 100

df["ops_var"] = ops.rolling_var(df["x"].to_numpy(), window)
df["var"] = df["x"].rolling(window, min_periods=1).var()

df["ops_skew"] = ops.rolling_skew(df["x"].to_numpy(), window)
df["skew"] = df["x"].rolling(window, min_periods=1).skew()

df["ops_kurt"] = ops.rolling_kurt(df["x"].to_numpy(), window)
df["kurt"] = df["x"].rolling(window, min_periods=1).kurt()
df

,x,ops_var,var,ops_skew,skew,ops_kurt,kurt
0,19.151945,NaN,NaN,NaN,NaN,NaN,NaN
1,62.210877,927.035815,927.035815,NaN,NaN,NaN,NaN
2,43.772774,466.703416,466.703416,-0.425383,-0.425383,NaN,NaN
3,78.535858,650.137226,650.137226,-0.398662,-0.398662,-0.733057,-0.733057
4,77.997581,634.265136,634.265136,-0.836560,-0.836560,-0.540840,-0.540840
...,...,...,...,...,...,...,...
9999995,11.115537,774.288441,774.288441,-0.059755,-0.059755,-1.172307,-1.172307
9999996,49.682310,772.976979,772.976979,-0.069681,-0.069681,-1.165180,-1.165180
9999997,19.468962,760.502498,760.502498,-0.051712,-0.051712,-1.176311,-1.176311
9999998,82.011629,760.534664,760.534664,-0.091870,-0.091870,-1.169913,-1.169913


In [4]:
print("Variance Max Error: ", (df["var"] - df["ops_var"]).abs().max())
print("Skewness Max Error: ", (df["skew"] - df["ops_skew"]).abs().max())
print("Kurtosis Max Error: ", (df["kurt"] - df["ops_kurt"]).abs().max())

Variance Max Error:  6.135678631835617e-10
Skewness Max Error:  2.5528190672474693e-13
Kurtosis Max Error:  2.2812862709997717e-12


## Test Time Series Rolling Moments

In [5]:
import duckdb

ORDER_TYPE_MARKET = 1
ORDER_TYPE_LIMIT = 2
ORDER_TYPE_BEST = 3
BS_FLAG_BUY = 1
BS_FLAG_SELL = 2
MATCH_TYPE_DEAL = 1
MATCH_TYPE_DEL = 2
INNER_FLAG_OUT = 1
INNER_FLAG_IN = 2

data_dir = "../data"

con = duckdb.connect()

con.sql("""
    create or replace macro time2ms(x) as
        ((x // 10000000)::int * 3600000) +   -- hours → ms
        ((x // 100000 % 100)::int * 60000) + -- minutes → ms
        ((x // 1000 % 100)::int * 1000) +    -- seconds → ms
        (x % 1000);                          -- milliseconds        
""")

df = con.sql(f"""
    select stockid, date, seq_num, time,
        time2ms(time)::uint32 as time_ms, qty, price,
    from '{data_dir}/deals_20260119.parquet'
    where match_type={MATCH_TYPE_DEAL}
        and time>=93000000 and time<145700000
""").df()
df["dt"] = pd.to_datetime(
    df["date"].astype(str) + df["time"].astype(str).str.zfill(9),
    format="%Y%m%d%H%M%S%f",
)

# add some NaNs for testing
# df.loc[[1, 10, 100, 1000], "qty"] = np.nan
df

,stockid,date,seq_num,time,time_ms,qty,price,dt
0,2270,20260119,904326,93000040,34200040,800,30000,2026-01-19 09:30:00.040
1,2270,20260119,906912,93000040,34200040,100,30020,2026-01-19 09:30:00.040
2,2270,20260119,906913,93000040,34200040,300,30020,2026-01-19 09:30:00.040
3,2270,20260119,906914,93000040,34200040,1000,30020,2026-01-19 09:30:00.040
4,2270,20260119,907098,93000040,34200040,100,30020,2026-01-19 09:30:00.040
...,...,...,...,...,...,...,...,...
19326,2270,20260119,57066672,145627040,53787040,800,32490,2026-01-19 14:56:27.040
19327,2270,20260119,57074135,145628100,53788100,400,32490,2026-01-19 14:56:28.100
19328,2270,20260119,57102109,145631610,53791610,800,32490,2026-01-19 14:56:31.610
19329,2270,20260119,57169567,145640350,53800350,400,32490,2026-01-19 14:56:40.350


In [6]:
field = "price"
window = 10
df[f"pd_{field}_var"] = df[field].rolling(window, min_periods=1).var()
df[f"ops_{field}_var"] = ops.rolling_var(
    df[field].to_numpy(),
    window,
)
print("Var Max Error: ", (df[f"pd_{field}_var"] - df[f"ops_{field}_var"]).abs().max())
df

Var Max Error:  4.704929779109079e-10


,stockid,date,seq_num,time,time_ms,qty,price,dt,pd_price_var,ops_price_var
0,2270,20260119,904326,93000040,34200040,800,30000,2026-01-19 09:30:00.040,NaN,NaN
1,2270,20260119,906912,93000040,34200040,100,30020,2026-01-19 09:30:00.040,200.000000,200.000000
2,2270,20260119,906913,93000040,34200040,300,30020,2026-01-19 09:30:00.040,133.333333,133.333333
3,2270,20260119,906914,93000040,34200040,1000,30020,2026-01-19 09:30:00.040,100.000000,100.000000
4,2270,20260119,907098,93000040,34200040,100,30020,2026-01-19 09:30:00.040,80.000000,80.000000
...,...,...,...,...,...,...,...,...,...,...
19326,2270,20260119,57066672,145627040,53787040,800,32490,2026-01-19 14:56:27.040,0.000000,0.000000
19327,2270,20260119,57074135,145628100,53788100,400,32490,2026-01-19 14:56:28.100,0.000000,0.000000
19328,2270,20260119,57102109,145631610,53791610,800,32490,2026-01-19 14:56:31.610,0.000000,0.000000
19329,2270,20260119,57169567,145640350,53800350,400,32490,2026-01-19 14:56:40.350,0.000000,0.000000


In [7]:
df.drop(columns=[f"pd_{field}_var", f"ops_{field}_var"], inplace=True)

In [8]:
window = 30_000
wstr = "30s"
min_obs = 1
field = "price"

df[f"pd_{field}_var_{wstr}"] = df.rolling(wstr, min_periods=min_obs, on="dt")[
    field
].var()
df[f"ops_{field}_var_{wstr}"] = ops.ts_rolling_var(
    df["time_ms"].to_numpy(),
    df["price"].to_numpy(),
    window,
    min_obs,
)

df[f"pd_{field}_skew_{wstr}"] = df.rolling(wstr, min_periods=min_obs, on="dt")[
    field
].skew()
df[f"ops_{field}_skew_{wstr}"] = ops.ts_rolling_skew(
    df["time_ms"].to_numpy(),
    df["price"].to_numpy(),
    window,
    min_obs,
)

df[f"pd_{field}_kurt_{wstr}"] = df.rolling(wstr, min_periods=min_obs, on="dt")[
    field
].kurt()
df[f"ops_{field}_kurt_{wstr}"] = ops.ts_rolling_kurt(
    df["time_ms"].to_numpy(),
    df["price"].to_numpy(),
    window,
    min_obs,
)

df

,stockid,date,seq_num,time,time_ms,qty,price,dt,pd_price_var_30s,ops_price_var_30s,pd_price_skew_30s,ops_price_skew_30s,pd_price_kurt_30s,ops_price_kurt_30s
0,2270,20260119,904326,93000040,34200040,800,30000,2026-01-19 09:30:00.040,NaN,NaN,NaN,NaN,NaN,NaN
1,2270,20260119,906912,93000040,34200040,100,30020,2026-01-19 09:30:00.040,200.000000,200.000000,NaN,NaN,NaN,NaN
2,2270,20260119,906913,93000040,34200040,300,30020,2026-01-19 09:30:00.040,133.333333,133.333333,-1.732051,-1.732051,NaN,NaN
3,2270,20260119,906914,93000040,34200040,1000,30020,2026-01-19 09:30:00.040,100.000000,100.000000,-2.000000,-2.000000,4.0,4.0
4,2270,20260119,907098,93000040,34200040,100,30020,2026-01-19 09:30:00.040,80.000000,80.000000,-2.236068,-2.236068,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19326,2270,20260119,57066672,145627040,53787040,800,32490,2026-01-19 14:56:27.040,0.000000,0.000000,0.000000,NaN,-3.0,NaN
19327,2270,20260119,57074135,145628100,53788100,400,32490,2026-01-19 14:56:28.100,0.000000,0.000000,0.000000,NaN,-3.0,NaN
19328,2270,20260119,57102109,145631610,53791610,800,32490,2026-01-19 14:56:31.610,0.000000,0.000000,0.000000,NaN,-3.0,NaN
19329,2270,20260119,57169567,145640350,53800350,400,32490,2026-01-19 14:56:40.350,0.000000,0.000000,0.000000,NaN,-3.0,NaN


In [9]:
con.sql("""
    select *
    from df
    where 
        abs(pd_price_var_30s - ops_price_var_30s)>1e-6
        --abs(pd_price_skew_30s - ops_price_skew_30s)>1e-6
        --abs(pd_price_kurt_30s - ops_price_kurt_30s)>1e-6
""").pl()

stockid,date,seq_num,time,time_ms,qty,price,dt,pd_price_var_30s,ops_price_var_30s,pd_price_skew_30s,ops_price_skew_30s,pd_price_kurt_30s,ops_price_kurt_30s
u32,i32,i64,i32,u32,u32,i32,datetime[ns],f64,f64,f64,f64,f64,f64
